In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset=pd.read_csv("insurance_pre.csv")

In [3]:
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [4]:
#Nominal - One-Hot-Encoding
#used function is "get.dummies" - the reason they used for this naming??
# when two field is zero automatically even if we don't mention newyork- newyork will get 1
#Column expansion
#Three uniq values such as california, florida, newyork
#reason for the column name expansion - State_California (state is the original column name)

In [5]:
dataset=pd.get_dummies(dataset,drop_first=True)

In [6]:
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [7]:
#Now we are going to split the input and output dataset
#
#
#

In [8]:
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [9]:
independent=dataset[['age', 'bmi', 'children', 'sex_male', 'smoker_yes']]

In [10]:
independent 

,age,bmi,children,sex_male,smoker_yes
0,19,27.900,0,0,1
1,18,33.770,1,1,0
2,28,33.000,3,1,0
3,33,22.705,0,1,0
4,32,28.880,0,1,0
...,...,...,...,...,...
1333,50,30.970,3,1,0
1334,18,31.920,0,0,0
1335,18,36.850,0,0,0
1336,21,25.800,0,0,0


In [11]:
dependent=dataset[["charges"]]

In [12]:
dependent

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520
...,...
1333,10600.54830
1334,2205.98080
1335,1629.83350
1336,2007.94500


In [13]:
#split into training set and test test
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(independent, dependent,test_size=0.30,random_state=0)

In [14]:
X_train

,age,bmi,children,sex_male,smoker_yes
1163,18,28.215,0,0,0
196,39,32.800,0,0,0
438,52,46.750,5,0,0
183,44,26.410,0,0,0
1298,33,27.455,2,1,0
...,...,...,...,...,...
763,27,26.030,0,1,0
835,42,35.970,2,1,0
1216,40,25.080,0,1,0
559,19,35.530,0,1,0


In [15]:
#Standardization


In [16]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()

X_train=sc.fit_transform(X_train)
X_test=sc.transform(X_test)

In [17]:
X_train

array([[-1.5330973 , -0.40713453, -0.89833872, -0.97676557, -0.50466988],
       [-0.03364163,  0.32855417, -0.89833872, -0.97676557, -0.50466988],
       [ 0.89459283,  2.56690911,  3.25603402, -0.97676557, -0.50466988],
       ...,
       [ 0.03776102, -0.91016269, -0.89833872,  1.02378711, -0.50466988],
       [-1.46169465,  0.76659782, -0.89833872,  1.02378711, -0.50466988],
       [-0.46205754, -1.96596021, -0.06746417, -0.97676557, -0.50466988]])

In [18]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
param_grid= {'criterion':['squared_error','poisson','absolute_error','friedman_mse'], 'max_features':[None,'sqrt','log2'],'splitter':['best','random']}

grid= GridSearchCV(DecisionTreeRegressor(), param_grid, refit=True, verbose=3, n_jobs=-1)

#fitting model

grid.fit(X_train,y_train)


Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(estimator=DecisionTreeRegressor(), n_jobs=-1,
             param_grid={'criterion': ['squared_error', 'poisson',
                                       'absolute_error', 'friedman_mse'],
                         'max_features': [None, 'sqrt', 'log2'],
                         'splitter': ['best', 'random']},
             verbose=3)

In [19]:
# print best parameter after tuning
# print (grid.best_params_)

re=grid.cv_results_

# print(re)

grid_predictions= grid.predict(X_test)

# print classification report

from sklearn.metrics import r2_score

r_score=r2_score(y_test,grid_predictions)

print("The R_score value for best parameter {}:" .format(grid.best_params_))

The R_score value for best parameter {'criterion': 'friedman_mse', 'max_features': None, 'splitter': 'best'}:


In [20]:
table=pd.DataFrame.from_dict(re)

In [21]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.031967,0.000360,0.002951,0.005903,squared_error,None,best,"{'criterion': 'squared_error', 'max_features':...",0.736924,0.537211,0.768683,0.625001,0.668540,0.667271,0.082268,2
1,0.026003,0.006566,0.002768,0.004129,squared_error,None,random,"{'criterion': 'squared_error', 'max_features':...",0.655453,0.699147,0.663848,0.446009,0.675409,0.627973,0.092162,12
2,0.017804,0.007309,0.013339,0.006742,squared_error,sqrt,best,"{'criterion': 'squared_error', 'max_features':...",0.778946,0.642923,0.649061,0.503984,0.599298,0.634842,0.088793,11
3,0.014260,0.010327,0.005424,0.005798,squared_error,sqrt,random,"{'criterion': 'squared_error', 'max_features':...",0.647247,0.493339,0.667739,0.657053,0.557603,0.604596,0.068036,15
4,0.016026,0.000824,0.009723,0.007979,squared_error,log2,best,"{'criterion': 'squared_error', 'max_features':...",0.696808,0.601285,0.585683,0.561052,0.599229,0.608811,0.046277,14
5,0.009314,0.007616,0.009447,0.007714,squared_error,log2,random,"{'criterion': 'squared_error', 'max_features':...",0.292481,0.513382,0.533981,0.593578,0.569528,0.500590,0.107691,24
6,0.028341,0.006274,0.006296,0.007711,poisson,None,best,"{'criterion': 'poisson', 'max_features': None,...",0.711119,0.596598,0.713418,0.592101,0.616843,0.646016,0.054738,7
7,0.017021,0.002616,0.005863,0.003160,poisson,None,random,"{'criterion': 'poisson', 'max_features': None,...",0.740171,0.645628,0.648588,0.672743,0.615913,0.664609,0.041866,3
8,0.015669,0.006118,0.006666,0.008164,poisson,sqrt,best,"{'criterion': 'poisson', 'max_features': 'sqrt...",0.664605,0.601931,0.687226,0.580993,0.685308,0.644012,0.044134,8
9,0.017244,0.001390,0.006256,0.006398,poisson,sqrt,random,"{'criterion': 'poisson', 'max_features': 'sqrt...",0.638148,0.479833,0.594371,0.559144,0.484920,0.551283,0.061601,20


In [22]:
age_input=float(input("Age:"))
bmi_input=float(input("BMI:"))
children_input=float(input("Children:"))
sex_male_input=int(input("Sex Male 0 or 1:"))
smoker_yes_input=int(input("Smoker Yes 0 or 1:"))


Age:43
BMI:21
Children:2
Sex Male 0 or 1:0
Smoker Yes 0 or 1:1


In [23]:
Future_Prediction=grid.predict([[age_input,bmi_input,children_input,sex_male_input,smoker_yes_input]])
print("Future_Prediction={}". format(Future_Prediction))

Future_Prediction=[63770.42801]
